In [1]:
# =============================================================================
# DKTC 전처리 파이프라인 (모델/토크나이저 독립)
# 목적: 최대한 깨끗하고 풍부한 학습 데이터를 만들어서 CSV로 저장
# 출력: data/train_processed.csv, data/val_processed.csv
# =============================================================================

import pandas as pd
import numpy as np
import re
import random
from pathlib import Path
from sklearn.model_selection import train_test_split

random.seed(42)
np.random.seed(42)

In [2]:
# ─────────────────────────────────────────────
# 1. 데이터 로드
# ─────────────────────────────────────────────
url = "https://raw.githubusercontent.com/tunib-ai/DKTC/main/data/train.csv"
raw_df = pd.read_csv(url, header=None, names=["id", "label_name", "conversation"])

# 헤더 행 제거
raw_df = raw_df[raw_df["id"] != "idx"].reset_index(drop=True)

label_map = {
    "협박 대화": 0,
    "갈취 대화": 1,
    "직장 내 괴롭힘 대화": 2,
    "기타 괴롭힘 대화": 3
}
raw_df["label"] = raw_df["label_name"].map(label_map).astype(int)
raw_df["id"] = raw_df["id"].astype(int)

print(f"raw_df.shape: {raw_df.shape}")
print(raw_df["label_name"].value_counts())

raw_df.shape: (3950, 4)
label_name
기타 괴롭힘 대화      1094
갈취 대화           981
직장 내 괴롭힘 대화     979
협박 대화           896
Name: count, dtype: int64


In [3]:
raw_df

,id,label_name,conversation,label
0,0,협박 대화,지금 너 스스로를 죽여달라고 애원하는 것인가?\n 아닙니다. 죄송합니다.\n 죽을 ...,0
1,1,협박 대화,길동경찰서입니다.\n9시 40분 마트에 폭발물을 설치할거다.\n네?\n똑바로 들어 ...,0
2,2,기타 괴롭힘 대화,너 되게 귀여운거 알지? 나보다 작은 남자는 첨봤어.\n그만해. 니들 놀리는거 재미...,3
3,3,갈취 대화,어이 거기\n예??\n너 말이야 너. 이리 오라고\n무슨 일.\n너 옷 좋아보인다?...,1
4,4,갈취 대화,저기요 혹시 날이 너무 뜨겁잖아요? 저희 회사에서 이 선크림 파는데 한 번 손등에 ...,1
...,...,...,...,...
3945,3945,기타 괴롭힘 대화,준하야 넌 대가리가 왜이렇게 크냐?\n내 머리가 뭐.\n밥먹으면 대가리만 크냐 너는...,3
3946,3946,갈취 대화,내가 지금 너 아들 김길준 데리고 있어. 살리고 싶으면 계좌에 1억만 보내\n예.?...,1
3947,3947,직장 내 괴롭힘 대화,나는 씨 같은 사람 보면 참 신기하더라. 어떻게 저렇게 살지.\n왜 그래. 들리겠어...,2
3948,3948,갈취 대화,누구맘대로 여기서 장사하래?\n이게 무슨일입니까?\n남의 구역에서 장사하려면 자릿세...,1


In [4]:
# ─────────────────────────────────────────────
# 2. 텍스트 정제 함수
# ─────────────────────────────────────────────
def clean_text(text: str) -> str:
    """한국어 대화체 텍스트 정제 파이프라인"""
    if not isinstance(text, str):
        return ""
    
    # (a) 연속 반복 문자 축소: ㅋㅋㅋㅋㅋ → ㅋㅋ, ㅎㅎㅎㅎ → ㅎㅎ
    #     위협적 반복도 축소: 죽여죽여죽여 → 죽여죽여
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    
    # (b) 자모 단독 반복 축소 (ㅋㅋㅋ, ㅎㅎㅎ, ㅠㅠㅠ 등)
    text = re.sub(r'([ㄱ-ㅎㅏ-ㅣ])\1{2,}', r'\1\1', text)
    
    # (c) 불필요한 특수문자 제거 (문장부호, 한글, 영문, 숫자, 공백, 줄바꿈 유지)
    text = re.sub(r'[^\w\s가-힣a-zA-Z0-9.,!?~\n]', ' ', text)
    
    # (d) 연속 공백 → 단일 공백
    text = re.sub(r'[ \t]+', ' ', text)
    
    # (e) 연속 줄바꿈 → 단일 줄바꿈 (대화 턴 구분 유지)
    text = re.sub(r'\n+', '\n', text)
    
    # (f) 앞뒤 공백 제거
    text = text.strip()
    
    return text


def normalize_conversation(text: str) -> str:
    """대화 구조 정규화 — 턴 구분을 명시적 토큰으로 변환"""
    if not isinstance(text, str):
        return ""
    
    # 줄바꿈으로 구분된 각 턴을 [SEP] 토큰으로 연결
    # → 토크나이저가 대화 경계를 인식할 수 있게 함
    turns = [t.strip() for t in text.split('\n') if t.strip()]
    
    # [SEP] 대신 " / "를 사용 — 대부분의 토크나이저에서 자연스럽게 처리됨
    # (모델 쪽에서 special token 추가하면 [SEP]로 변경 가능)
    return " [턴] ".join(turns)


In [5]:
# ─────────────────────────────────────────────
# 3. 대화 메타 피처 추출
# ─────────────────────────────────────────────
def extract_meta_features(text: str) -> dict:
    """대화에서 구조적 특성 추출 (분석용 + 필터링용)"""
    turns = [t.strip() for t in text.split('\n') if t.strip()]
    lengths = [len(t) for t in turns]
    
    return {
        "n_turns": len(turns),                          # 대화 턴 수
        "total_chars": sum(lengths),                    # 전체 문자 수
        "avg_turn_len": np.mean(lengths) if lengths else 0,  # 평균 턴 길이
        "max_turn_len": max(lengths) if lengths else 0,      # 최장 턴 길이
    }

In [6]:
# ─────────────────────────────────────────────
# 4. 텍스트 정제 적용 + 메타 피처 추출
# ─────────────────────────────────────────────
raw_df["conversation_clean"] = raw_df["conversation"].apply(clean_text)
raw_df["conversation_norm"] = raw_df["conversation_clean"].apply(normalize_conversation)

# 메타 피처 (정제 전 원본 기준으로 추출)
meta = raw_df["conversation"].apply(extract_meta_features).apply(pd.Series)
raw_df = pd.concat([raw_df, meta], axis=1)

print(f"\n정제 후 통계:")
print(raw_df.groupby("label_name")[["n_turns", "total_chars", "avg_turn_len"]].mean().round(1))


정제 후 통계:
             n_turns  total_chars  avg_turn_len
label_name                                     
갈취 대화           10.5        204.7          19.4
기타 괴롭힘 대화       10.2        199.0          19.4
직장 내 괴롭힘 대화     10.4        226.2          21.7
협박 대화           10.3        234.6          22.6


In [7]:
print(f"{raw_df.shape}")
print(raw_df["label_name"].value_counts())

(3950, 10)
label_name
기타 괴롭힘 대화      1094
갈취 대화           981
직장 내 괴롭힘 대화     979
협박 대화           896
Name: count, dtype: int64


In [8]:
raw_df

,id,label_name,conversation,label,conversation_clean,conversation_norm,n_turns,total_chars,avg_turn_len,max_turn_len
0,0,협박 대화,지금 너 스스로를 죽여달라고 애원하는 것인가?\n 아닙니다. 죄송합니다.\n 죽을 ...,0,지금 너 스스로를 죽여달라고 애원하는 것인가?\n 아닙니다. 죄송합니다.\n 죽을 ...,지금 너 스스로를 죽여달라고 애원하는 것인가? [턴] 아닙니다. 죄송합니다. [턴]...,10.0,224.0,22.400000,43.0
1,1,협박 대화,길동경찰서입니다.\n9시 40분 마트에 폭발물을 설치할거다.\n네?\n똑바로 들어 ...,0,길동경찰서입니다.\n9시 40분 마트에 폭발물을 설치할거다.\n네?\n똑바로 들어 ...,길동경찰서입니다. [턴] 9시 40분 마트에 폭발물을 설치할거다. [턴] 네? [턴...,10.0,177.0,17.700000,39.0
2,2,기타 괴롭힘 대화,너 되게 귀여운거 알지? 나보다 작은 남자는 첨봤어.\n그만해. 니들 놀리는거 재미...,3,너 되게 귀여운거 알지? 나보다 작은 남자는 첨봤어.\n그만해. 니들 놀리는거 재미...,너 되게 귀여운거 알지? 나보다 작은 남자는 첨봤어. [턴] 그만해. 니들 놀리는거...,10.0,207.0,20.700000,33.0
3,3,갈취 대화,어이 거기\n예??\n너 말이야 너. 이리 오라고\n무슨 일.\n너 옷 좋아보인다?...,1,어이 거기\n예??\n너 말이야 너. 이리 오라고\n무슨 일.\n너 옷 좋아보인다?...,어이 거기 [턴] 예?? [턴] 너 말이야 너. 이리 오라고 [턴] 무슨 일. [턴...,11.0,105.0,9.545455,20.0
4,4,갈취 대화,저기요 혹시 날이 너무 뜨겁잖아요? 저희 회사에서 이 선크림 파는데 한 번 손등에 ...,1,저기요 혹시 날이 너무 뜨겁잖아요? 저희 회사에서 이 선크림 파는데 한 번 손등에 ...,저기요 혹시 날이 너무 뜨겁잖아요? 저희 회사에서 이 선크림 파는데 한 번 손등에 ...,12.0,449.0,37.416667,63.0
...,...,...,...,...,...,...,...,...,...,...
3945,3945,기타 괴롭힘 대화,준하야 넌 대가리가 왜이렇게 크냐?\n내 머리가 뭐.\n밥먹으면 대가리만 크냐 너는...,3,준하야 넌 대가리가 왜이렇게 크냐?\n내 머리가 뭐.\n밥먹으면 대가리만 크냐 너는...,준하야 넌 대가리가 왜이렇게 크냐? [턴] 내 머리가 뭐. [턴] 밥먹으면 대가리만...,11.0,223.0,20.272727,37.0
3946,3946,갈취 대화,내가 지금 너 아들 김길준 데리고 있어. 살리고 싶으면 계좌에 1억만 보내\n예.?...,1,내가 지금 너 아들 김길준 데리고 있어. 살리고 싶으면 계좌에 1억만 보내\n예.?...,내가 지금 너 아들 김길준 데리고 있어. 살리고 싶으면 계좌에 1억만 보내 [턴] ...,10.0,214.0,21.400000,45.0
3947,3947,직장 내 괴롭힘 대화,나는 씨 같은 사람 보면 참 신기하더라. 어떻게 저렇게 살지.\n왜 그래. 들리겠어...,2,나는 씨 같은 사람 보면 참 신기하더라. 어떻게 저렇게 살지.\n왜 그래. 들리겠어...,나는 씨 같은 사람 보면 참 신기하더라. 어떻게 저렇게 살지. [턴] 왜 그래. 들...,11.0,291.0,26.454545,63.0
3948,3948,갈취 대화,누구맘대로 여기서 장사하래?\n이게 무슨일입니까?\n남의 구역에서 장사하려면 자릿세...,1,누구맘대로 여기서 장사하래?\n이게 무슨일입니까?\n남의 구역에서 장사하려면 자릿세...,누구맘대로 여기서 장사하래? [턴] 이게 무슨일입니까? [턴] 남의 구역에서 장사하...,10.0,216.0,21.600000,43.0


In [9]:
# ─────────────────────────────────────────────
# 5. 이상치 / 저품질 데이터 필터링
# ─────────────────────────────────────────────
before_filter = len(raw_df)

# (a) 빈 텍스트 제거
raw_df = raw_df[raw_df["conversation_clean"].str.len() > 0]

# (b) 너무 짧은 대화 제거 (1턴짜리, 10자 미만)
raw_df = raw_df[~((raw_df["n_turns"] <= 1) & (raw_df["total_chars"] < 10))]

# (c) 완전 중복 제거
raw_df = raw_df.drop_duplicates(subset=["conversation_clean"], keep="first")

# (d) 유사 중복 제거 (정규화 후 동일)
raw_df = raw_df.drop_duplicates(subset=["conversation_norm"], keep="first")

print(f"\n필터링: {before_filter} → {len(raw_df)} ({before_filter - len(raw_df)}개 제거)")
print(raw_df["label_name"].value_counts())



필터링: 3950 → 3845 (105개 제거)
label_name
기타 괴롭힘 대화      1010
갈취 대화           973
직장 내 괴롭힘 대화     970
협박 대화           892
Name: count, dtype: int64


In [ ]:
# ─────────────────────────────────────────────
# 6. 일반 대화 데이터 생성 _ 외부데이터 가져오기
# ─────────────────────────────────────────────
# 핵심: 다양한 도메인, 다양한 턴 수, 다양한 톤으로 생성

# 예시 1: AI Hub 일상대화 등 CSV 파일
external_df = pd.read_csv("data/external_normal.csv")  # 텍스트 컬럼명에 맞게 수정
normal_df = pd.DataFrame({
    "id": range(10000, 10000 + len(external_df)),
    "label_name": "일반 대화",
    "conversation": external_df["your_text_column"],  # ← 실제 컬럼명으로 변경
    "label": 4
})

# 예시 2: JSON 파일 (AI Hub 등)
import json
with open("data/external_normal.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# AI Hub 형식이면 대화 턴을 \n으로 합쳐야 함
conversations = []
for item in data:
    turns = [utt["utterance"] for utt in item["utterances"]]  # ← 실제 키에 맞게
    conversations.append("\n".join(turns))

normal_df = pd.DataFrame({
    "id": range(10000, 10000 + len(conversations)),
    "label_name": "일반 대화",
    "conversation": conversations,
    "label": 4
})

FileNotFoundError: [Errno 2] No such file or directory: 'data/external_normal.csv'

In [ ]:
# ─────────────────────────────────────────────
# 7. 데이터 증강 (텍스트 레벨)
# ─────────────────────────────────────────────
# (augment_turn_shuffle, augment_random_deletion, augment_turn_drop, augment_data 함수는 그대로)

# ── 클래스 균형 조정 ──
threat_max = train_df[train_df["label"] != 4]["label"].value_counts().max()
normal_count = len(train_df[train_df["label"] == 4])

if normal_count > threat_max * 2:
    normal_subset = train_df[train_df["label"] == 4].sample(n=threat_max, random_state=42)
    train_df = pd.concat([train_df[train_df["label"] != 4], normal_subset], ignore_index=True)
    target = threat_max
else:
    target = int(train_df["label"].value_counts().median())

print(f"증강 목표: 클래스당 {target}개")

# ── 증강 실행 ──
train_aug = augment_data(train_df, target_per_class=target)

print(f"증강 후 학습 데이터:")
print(train_aug["label_name"].value_counts())

In [ ]:
# ─────────────────────────────────────────────
# 8. 데이터 합치기 + 증강 + 분할
# ─────────────────────────────────────────────

# (a) 원본 4클래스 + 일반대화 합치기
combined_df = pd.concat([raw_df, normal_df], ignore_index=True)
print(f"\n[5] 합친 데이터: {len(combined_df)}개")
print(combined_df["label_name"].value_counts())

# (b) train/val 분할 (증강 전에 분할 — 데이터 누출 방지!)
#     증강 후에 분할하면 원본-증강 쌍이 train/val에 나뉠 수 있음
train_df, val_df = train_test_split(
    combined_df,
    test_size=0.2,
    stratify=combined_df["label"],
    random_state=42
)

print(f"\n[6] 분할 (증강 전):")
print(f"  학습: {len(train_df)}")
print(f"  검증: {len(val_df)}")

# (c) 학습 데이터만 증강 (검증 데이터는 절대 건드리지 않음)
#     가장 많은 클래스의 수에 맞춤
max_class_count = train_df["label"].value_counts().max()
# 일반대화는 원래 적으므로, 다른 클래스의 중앙값 정도로 설정
target = int(train_df["label"].value_counts().median())
print(f"\n[7] 증강 목표: 클래스당 {target}개")

train_aug = augment_data(train_df, target_per_class=target)

print(f"\n[8] 증강 후 학습 데이터:")
print(train_aug["label_name"].value_counts())


In [ ]:
# ─────────────────────────────────────────────
# 9. 최종 정제: 학습에 사용할 컬럼만 선택 + 저장
# ─────────────────────────────────────────────
# conversation_norm 을 최종 학습 텍스트로 사용

output_cols = ["conversation_norm", "label", "label_name", "n_turns", "total_chars"]

train_final = train_aug[output_cols].rename(columns={"conversation_norm": "text"})
val_final = val_df[output_cols].rename(columns={"conversation_norm": "text"})

# 빈 텍스트 최종 확인
train_final = train_final[train_final["text"].str.len() > 0].reset_index(drop=True)
val_final = val_final[val_final["text"].str.len() > 0].reset_index(drop=True)

# 저장
Path("data").mkdir(exist_ok=True)
train_final.to_csv("data/train_processed.csv", index=False)
val_final.to_csv("data/val_processed.csv", index=False)

print(f"\n{'='*60}")
print(f"[최종 결과]")
print(f"  학습: {len(train_final)}개")
print(f"  검증: {len(val_final)}개")
print(f"  클래스: {sorted(train_final['label'].unique())}")
print(f"\n  학습 클래스 분포:")
print(train_final["label_name"].value_counts().to_string())
print(f"\n  검증 클래스 분포:")
print(val_final["label_name"].value_counts().to_string())
print(f"\n  저장 경로: data/train_processed.csv, data/val_processed.csv")
print(f"  텍스트 컬럼: 'text' (conversation_norm)")
print(f"  라벨 컬럼: 'label' (0~4)")
print(f"{'='*60}")


In [ ]:
# ─────────────────────────────────────────────
# 10. 전처리 품질 리포트
# ─────────────────────────────────────────────
print(f"\n[품질 리포트]")

# 텍스트 길이 분포
print(f"\n  텍스트 길이 (글자 수):")
for label_name in train_final["label_name"].unique():
    subset = train_final[train_final["label_name"] == label_name]["text"]
    lengths = subset.str.len()
    print(f"    {label_name}: mean={lengths.mean():.0f}, "
          f"median={lengths.median():.0f}, "
          f"min={lengths.min()}, max={lengths.max()}")

# 토큰 수 추정 (대략 한글 1글자 ≈ 1~2 토큰)
print(f"\n  ⚠️ max_length 설정 가이드:")
print(f"     텍스트 최대 길이: {train_final['text'].str.len().max()} 글자")
print(f"     95퍼센타일: {train_final['text'].str.len().quantile(0.95):.0f} 글자")
print(f"     → klue/roberta-base 기준 max_length=256 권장")
print(f"        (128이면 긴 대화가 잘림)")
